# Video Intelligence & Human Detection Platform

Privacy-conscious video analytics: detect people, assign temporary anonymous track IDs, measure sampled-frame occupancy, and flag configurable scene events.

**Privacy boundary:** person detection only. No face recognition, identity matching, or sensitive-attribute inference.

## 1. Install dependencies

Run in a fresh Jupyter environment. Restart the kernel if prompted.

In [ ]:
%pip install -q ultralytics opencv-python-headless pandas matplotlib

## 2. Imports and configuration

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ultralytics import YOLO

VIDEO_PATH = "sample_video.mp4"   # Replace with an authorized local video.
MODEL_NAME = "yolov8n.pt"         # Downloads weights on first use.
CONFIDENCE_THRESHOLD = 0.35
FRAME_SAMPLE_EVERY = 5
MAX_FRAMES = None
CROWD_THRESHOLD = 5
ROI = None  # Optional pixel rectangle: (x1, y1, x2, y2)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

model = YOLO(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

## 3. Lightweight anonymous tracker

Greedy IoU matching is used for demonstration. IDs are temporary and may switch during occlusion or fast movement.

In [ ]:
def box_iou(a, b):
    xl, yt = max(a[0], b[0]), max(a[1], b[1])
    xr, yb = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xr-xl) * max(0, yb-yt)
    aa = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    ab = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (aa+ab-inter) if aa+ab-inter else 0.0

class IoUTracker:
    def __init__(self, threshold=0.25, max_missed=2):
        self.threshold, self.max_missed = threshold, max_missed
        self.next_id, self.tracks = 1, {}

    def update(self, boxes):
        candidates = [(box_iou(t["box"], b), tid, i)
                      for tid, t in self.tracks.items()
                      for i, b in enumerate(boxes)
                      if box_iou(t["box"], b) >= self.threshold]
        candidates.sort(reverse=True)
        assigned, used_t, used_d = {}, set(), set()
        for score, tid, i in candidates:
            if tid not in used_t and i not in used_d:
                assigned[i] = tid
                used_t.add(tid); used_d.add(i)
        for i in range(len(boxes)):
            if i not in assigned:
                assigned[i] = self.next_id
                self.next_id += 1
        updated = {assigned[i]: {"box": b, "missed": 0} for i, b in enumerate(boxes)}
        for tid, track in self.tracks.items():
            if tid not in updated and tid not in used_t:
                track["missed"] += 1
                if track["missed"] <= self.max_missed:
                    updated[tid] = track
        self.tracks = updated
        return assigned

def center(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)

def in_roi(point, roi):
    if roi is None: return False
    x, y = point; x1, y1, x2, y2 = roi
    return x1 <= x <= x2 and y1 <= y <= y2

## 4. Analyze video

The notebook samples frames, runs person-only inference (COCO class 0), tracks detections, and records crowd/zone events.

In [ ]:
video_path = Path(VIDEO_PATH)
if not video_path.exists():
    raise FileNotFoundError(f"Video not found: {video_path.resolve()}. Set VIDEO_PATH to an authorized video.")

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened(): raise RuntimeError(f"Could not open {video_path}")
fps = cap.get(cv2.CAP_PROP_FPS) or 0
width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
tracker = IoUTracker()
detections, events, annotated_frames = [], [], []
previous_roi_ids = set()
frame_idx = 0
started = time.time()

while True:
    ok, frame = cap.read()
    if not ok: break
    idx = frame_idx
    frame_idx += 1
    if idx % FRAME_SAMPLE_EVERY: continue
    if MAX_FRAMES is not None and len(annotated_frames) >= MAX_FRAMES: break

    pred = model.predict(frame, conf=CONFIDENCE_THRESHOLD, classes=[0], verbose=False)[0]
    boxes, confs = [], []
    if pred.boxes is not None:
        for item in pred.boxes:
            boxes.append(item.xyxy[0].cpu().numpy().astype(float).tolist())
            confs.append(float(item.conf[0].cpu().numpy()))

    assignment = tracker.update(boxes)
    timestamp = idx / fps if fps else None
    roi_ids = set()
    canvas = frame.copy()

    for i, box in enumerate(boxes):
        tid = assignment[i]
        x1,y1,x2,y2 = map(int, box)
        cx,cy = center(box)
        if in_roi((cx,cy), ROI): roi_ids.add(tid)
        cv2.rectangle(canvas, (x1,y1), (x2,y2), (40,190,80), 2)
        cv2.putText(canvas, f"Person / ID {tid}", (x1,max(20,y1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, .55, (40,190,80), 2)
        detections.append({"frame":idx,"timestamp_sec":timestamp,"track_id":tid,
                           "confidence":confs[i],"x1":box[0],"y1":box[1],
                           "x2":box[2],"y2":box[3],"center_x":cx,"center_y":cy})

    if len(boxes) >= CROWD_THRESHOLD:
        events.append({"frame":idx,"timestamp_sec":timestamp,"event_type":"crowd_threshold",
                       "details":f"{len(boxes)} people detected; threshold {CROWD_THRESHOLD}"})
    for tid in roi_ids - previous_roi_ids:
        events.append({"frame":idx,"timestamp_sec":timestamp,"event_type":"zone_entry",
                       "details":f"Temporary track {tid} entered configured ROI"})
    previous_roi_ids = roi_ids

    cv2.putText(canvas, f"People detected: {len(boxes)}", (15,30),
                cv2.FONT_HERSHEY_SIMPLEX, .8, (0,220,255), 2)
    if ROI is not None:
        x1,y1,x2,y2 = ROI
        cv2.rectangle(canvas,(x1,y1),(x2,y2),(255,160,0),2)
    annotated_frames.append(canvas)

cap.release()
detections_df = pd.DataFrame(detections)
events_df = pd.DataFrame(events)
print("Source frames:", frame_idx, "| sampled frames:", len(annotated_frames),
      "| elapsed seconds:", round(time.time()-started,1))
display(detections_df.head())
display(events_df.head())

## 5. Occupancy timeline and summary

In [ ]:
if detections_df.empty:
    occupancy_df = pd.DataFrame(columns=["frame","timestamp_sec","people_count"])
else:
    occupancy_df = detections_df.groupby(["frame","timestamp_sec"], dropna=False)["track_id"].nunique().reset_index(name="people_count")

if not occupancy_df.empty:
    plt.figure(figsize=(12,4))
    plt.plot(occupancy_df.timestamp_sec, occupancy_df.people_count, marker="o")
    plt.axhline(CROWD_THRESHOLD, linestyle="--", label=f"Threshold {CROWD_THRESHOLD}")
    plt.title("People detected per sampled frame")
    plt.xlabel("Video time (seconds)"); plt.ylabel("People detected")
    plt.legend(); plt.grid(alpha=.3); plt.show()

summary = {
    "video_file": str(video_path), "fps": fps, "width": width, "height": height,
    "source_total_frames": total_frames, "sample_interval_frames": FRAME_SAMPLE_EVERY,
    "sampled_frames_analyzed": len(occupancy_df),
    "peak_people_detected": int(occupancy_df.people_count.max()) if not occupancy_df.empty else 0,
    "detection_records": len(detections_df), "event_count": len(events_df),
    "event_types": events_df.event_type.value_counts().to_dict() if not events_df.empty else {}
}
summary

## 6. Export results

In [ ]:
# Annotated output is a sampled review clip, not a real-time stream.
annotated_path = OUTPUT_DIR / "annotated_sampled_video.mp4"
if annotated_frames:
    out_fps = max(1.0, fps / FRAME_SAMPLE_EVERY if fps else 5.0)
    writer = cv2.VideoWriter(str(annotated_path), cv2.VideoWriter_fourcc(*"mp4v"),
                             out_fps, (width,height))
    for f in annotated_frames: writer.write(f)
    writer.release()

detections_df.to_csv(OUTPUT_DIR / "detections.csv", index=False)
events_df.to_csv(OUTPUT_DIR / "events.csv", index=False)
occupancy_df.to_csv(OUTPUT_DIR / "occupancy_timeline.csv", index=False)
with open(OUTPUT_DIR / "summary.json","w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,default=str)
print("Saved outputs to:", OUTPUT_DIR.resolve())

## 7. Preview an annotated frame

In [ ]:
if annotated_frames:
    rgb = cv2.cvtColor(annotated_frames[min(2,len(annotated_frames)-1)], cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12,7)); plt.imshow(rgb); plt.axis("off"); plt.show()
else:
    print("No frames available.")

## 8. Extensions and responsible use

**Potential extensions**
- Replace greedy IoU matching with ByteTrack or BoT-SORT.
- Add line-crossing and dwell-time analytics.
- Build a Streamlit dashboard and batch-processing queue.
- Evaluate precision, recall, and mAP on a labeled validation set.
- Add role-based access, audit logs, secure storage, and retention controls.

**Responsible use**
- Analyze only footage you are authorized to process.
- Provide notice and comply with applicable privacy and surveillance laws.
- Minimize retention and restrict access to videos and exports.
- Treat alerts as review cues, not definitive safety conclusions.
- Do not add face recognition, identity matching, or sensitive-attribute inference.